# 09 - Evaluation

Six models built across Phases 4-6, each evaluated with the same Precision@10 / Recall@10
harness so the numbers are directly comparable. This notebook pulls every saved result
together into one leaderboard and states the actual conclusion — which model wins, and why,
grounded in what the feature importances and Phase 5 experiments already showed rather than
just the final numbers in isolation.

In [1]:
import pandas as pd
from pathlib import Path

processed_dir = Path.cwd().parent / "data" / "processed"

## Load every result file

In [2]:
all_results = pd.concat([
    pd.read_csv(processed_dir / "baseline_results.csv", index_col=0),
    pd.read_csv(processed_dir / "als_results.csv", index_col=0),
    pd.read_csv(processed_dir / "content_based_results.csv", index_col=0),
    pd.read_csv(processed_dir / "hybrid_results.csv", index_col=0),
    pd.read_csv(processed_dir / "ranking_model_results.csv", index_col=0),
])

leaderboard = all_results[["n_users_evaluated", "precision_at_k", "recall_at_k"]].sort_values(
    "precision_at_k", ascending=False
)
leaderboard

,n_users_evaluated,precision_at_k,recall_at_k
Ranking Model,26241.0,0.295263,0.350112
Personalized Frequency,131209.0,0.283836,0.329784
Hybrid,131209.0,0.276123,0.330263
Popularity,131209.0,0.072522,0.069843
Reorder Popularity,131209.0,0.072151,0.069616
ALS,131209.0,0.065520,0.098105
Content-Based,131209.0,0.002762,0.002989


`n_users_evaluated` is included deliberately, not just precision/recall — Ranking Model was
scored on 26,241 validation users (the 20% held out from training), everything else on the
full 131,209-user eval set. Same method, different sample size, so the comparison is fair in
how it's measured but not a perfectly matched sample.

## Conclusion

**Ranking Model wins** on both precision (0.2953) and recall (0.3501), ahead of Personalized
Frequency (0.2838 / 0.3298) and the Hybrid (0.2761 / 0.3303).

That gain isn't from a new signal, though — permutation importance in Phase 6 showed
`up_purchase_count` and `orders_since_last_purchase` account for roughly 85% of the model's
decisions, which is substantially the same reorder logic Personalized Frequency already runs
on. What the ranking model adds is refinement: it uses secondary features (`n_orders`,
`reorder_rate`) to break ties and edge cases that a simple frequency count can't, which is
enough to edge out both PF and the Hybrid without representing a fundamentally different
approach to the problem.

The bigger pattern across every model built in this project is the same one first seen in the
Phase 3 EDA: quick-commerce grocery shopping is driven overwhelmingly by habit. A 0.59 overall
reorder rate meant that any model built on frequency/reorder signal (Personalized Frequency,
the Hybrid, the Ranking Model) landed in the same tier, while collaborative filtering and
content-based similarity — built for discovering *new* relevant items — scored far lower
because that's a smaller, harder problem on this dataset, not because those approaches are
flawed. ALS's real value showed up in the overlap check (Phase 5), where it and content-based
surfaced almost entirely different products (0.0246 Jaccard) — useful for discovery/cross-sell
even where neither wins on reorder prediction alone.

For a production system, this suggests: use a frequency/reorder-based model (or the Ranking
Model, given the extra precision) for the primary "reorder" shelf, and keep ALS/content-based
as a separate discovery shelf rather than trying to force one model to do both jobs well.

In [3]:
leaderboard.to_csv(processed_dir / "final_leaderboard.csv")